In [2]:
!pip install biopython scikit-learn numpy pandas transformers torch -q


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\prana\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [12]:
import re
import torch
import numpy as np
import pandas as pd
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [5]:
def parse_fasta(filepath):
    headers, seqs = [], []
    for record in SeqIO.parse(filepath, "fasta"):
        headers.append(record.id)
        seqs.append(str(record.seq))
    return headers, seqs

signal_headers, signal_seqs = parse_fasta("Data/signal_filtered.fasta")
nonsignal_headers, nonsignal_seqs = parse_fasta("Data/nonsignal_filtered.fasta")

all_headers = signal_headers + nonsignal_headers
all_seqs    = signal_seqs + nonsignal_seqs
labels      = [1] * len(signal_seqs) + [0] * len(nonsignal_seqs)

print(f"Signal: {len(signal_seqs)}, Non-signal: {len(nonsignal_seqs)}, Total: {len(all_seqs)}")

Signal: 498, Non-signal: 474, Total: 972


In [6]:
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"

def get_physicochemical(seq):
    clean = ''.join([aa for aa in seq if aa in AMINO_ACIDS])
    if len(clean) < 5:
        return np.zeros(6)
    try:
        a = ProteinAnalysis(clean)
        return np.array([
            a.gravy(),                           # hydrophobicity
            a.isoelectric_point(),               # isoelectric point
            a.molecular_weight(),                # molecular weight
            a.instability_index(),               # instability index
            a.aromaticity(),                     # aromaticity
            a.secondary_structure_fraction()[0]  # helix propensity
        ])
    except:
        return np.zeros(6)

physchem = np.array([get_physicochemical(s) for s in all_seqs])
physchem_scaled = StandardScaler().fit_transform(physchem)
print("Physicochemical shape:", physchem_scaled.shape)  # (n, 6)

Physicochemical shape: (972, 6)


In [7]:
print("Loading BioBERT...")
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-v1.1")
model = AutoModel.from_pretrained("dmis-lab/biobert-v1.1")
model.eval()

bert_embeddings = []

for i, seq in enumerate(all_seqs):
    spaced_seq = ' '.join(seq[:500])
    inputs = tokenizer(spaced_seq, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
        emb = outputs.last_hidden_state[:, 0, :].numpy()
        bert_embeddings.append(emb[0])
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(all_seqs)} done")

bert_embeddings = np.array(bert_embeddings)
print("BioBERT shape:", bert_embeddings.shape)  # (n, 768)

Loading BioBERT...


C:\Users\prana\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\prana\.cache\huggingface\hub\models--dmis-lab--biobert-v1.1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|████

  100/972 done
  200/972 done
  300/972 done
  400/972 done
  500/972 done
  600/972 done
  700/972 done
  800/972 done
  900/972 done
BioBERT shape: (972, 768)


In [ ]:
# combine BERT (768) + physicochemical (6) = 774 features
X = np.hstack([bert_embeddings, physchem_scaled])
y = np.array(labels)

print(f"Final X shape: {X.shape}")  # (n, 774)

# save as CSV
df = pd.DataFrame(bert_embeddings, columns=[f'emb_{i}' for i in range(768)])
df['gravy']       = physchem[:, 0]
df['isoelectric'] = physchem[:, 1]
df['mol_weight']  = physchem[:, 2]
df['instability'] = physchem[:, 3]
df['aromaticity'] = physchem[:, 4]
df['helix']       = physchem[:, 5]
df['label']       = labels
df['id']          = all_headers
df.to_csv("Data/features_all.csv", index=False)

print(f"Saved {len(all_seqs)} sequences, {X.shape[1]} features each")

Final X shape: (972, 774)
Saved 972 sequences, 774 features each


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (777, 774), Test: (195, 774)


In [14]:
class CNNBiLSTMAttention(nn.Module):
    def __init__(self, input_dim=774):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )
        self.bilstm = nn.LSTM(
            input_size=387,   # after maxpool: 774//2
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )
        self.attention = nn.Linear(256, 1)
        self.fc = nn.Linear(256, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = x.unsqueeze(1)           
        x = self.cnn(x)              
        x = x.permute(0, 2, 1)      

        # BiLSTM
        x, _ = self.bilstm(x)       

        # Attention
        attn = torch.softmax(self.attention(x), dim=1)
        x = (x * attn).sum(dim=1)   

        # Output
        return self.sigmoid(self.fc(x)).squeeze()

model_dl = CNNBiLSTMAttention()
print(model_dl)

CNNBiLSTMAttention(
  (cnn): Sequential(
    (0): Conv1d(1, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): ReLU()
    (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (bilstm): LSTM(387, 128, num_layers=2, batch_first=True, bidirectional=True)
  (attention): Linear(in_features=256, out_features=1, bias=True)
  (fc): Linear(in_features=256, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [15]:
X_tr = torch.tensor(X_train, dtype=torch.float32)
y_tr = torch.tensor(y_train, dtype=torch.float32)
X_te = torch.tensor(X_test,  dtype=torch.float32)
y_te = torch.tensor(y_test,  dtype=torch.float32)

loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)

optimizer = torch.optim.Adam(model_dl.parameters(), lr=1e-3)
criterion = nn.BCELoss()

for epoch in range(20):
    model_dl.train()
    total_loss = 0
    for xb, yb in loader:
        optimizer.zero_grad()
        preds = model_dl(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/20 — Loss: {total_loss/len(loader):.4f}")

RuntimeError: input.size(-1) must be equal to input_size. Expected 387, got 64